# Verdict Workstation Gemma 4 Pilot
This notebook trains only on the 75/15 provider-neutral public-market pilot. It never loads the sealed 30-case evaluation set or personal portfolio data. Run it on a private signed-in Colab GPU session; artifacts remain in your private Google Drive and are never published.

In [ ]:
CONFIRMATION = input('Type I CONFIRM to mount private Drive and later upload the 75/15 public pilot files: ').strip()
if CONFIRMATION != 'I CONFIRM':
    raise RuntimeError('Explicit confirmation was not provided.')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!nvidia-smi

In [ ]:
from pathlib import Path
import subprocess
DRIVE_ROOT = Path('/content/drive/MyDrive/Verdict-Workstation/gemma-bakeoff')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import re
EXPECTED_COMMIT = input('Paste the 40-character reviewed Git commit for this pilot: ').strip()
if not re.fullmatch(r'[0-9a-f]{40}', EXPECTED_COMMIT):
    raise RuntimeError('A full 40-character Git commit is required.')
REPO = Path('/content/Verdict-Workstation')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jainmrigank/Verdict-Workstation.git', str(REPO)], check=True)
APP = REPO / 'New project' / 'stock-analysis-app'
import os
os.chdir(APP)
if subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Colab repository is dirty; start a fresh runtime before training.')
subprocess.run(['git', 'fetch', '-q', 'origin', EXPECTED_COMMIT], check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', EXPECTED_COMMIT], check=True)
subprocess.run(['python3', '-m', 'pip', 'install', '-q', '-r', 'notebooks/gemma_bakeoff_requirements.txt'], check=True)
subprocess.run(['python3', '-m', 'pip', 'freeze'], check=True, stdout=(DRIVE_ROOT / 'colab-packages.lock.txt').open('w'))
print('Working directory:', Path.cwd())
actual_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f'Expected {EXPECTED_COMMIT}, found {actual_commit}')
print('Repository commit:', actual_commit)

Upload exactly `pilot_train.jsonl` and `pilot_validation.jsonl`. The next cell rejects additional files and the CLI rejects a changed dataset hash, split leakage, or any sealed evaluation ID.

In [ ]:
from google.colab import files
uploaded = files.upload()
expected_names = {'pilot_train.jsonl', 'pilot_validation.jsonl'}
if set(uploaded) != expected_names:
    raise ValueError(f'Expected exactly {sorted(expected_names)}, received {sorted(uploaded)}')
target = APP / 'data' / 'training' / 'provider'
target.mkdir(parents=True, exist_ok=True)
for name, payload in uploaded.items():
    (target / name).write_bytes(payload)

In [ ]:
PILOT_HASH = '6a374ec58b9fd134de5cf1ccdeda33384ba24c02935c4af2b3e3d8a6bbd75c09'
common = [
    'python3', 'tools/gemma_bakeoff.py',
    '--data-root', str(target),
    '--output-root', str(DRIVE_ROOT),
    '--expected-dataset-hash', PILOT_HASH,
]
subprocess.run([*common, '--dry-run', '--model', 'gemma4-e4b'], check=True)

## E4B primary candidate
Preflight loads the 4-bit base model and derives the sequence limit from every rendered train and validation conversation. Training is not started unless the complete input fits without truncation.

In [ ]:
E4B_RUN = 'gemma4-e4b-r16-e2-seed560'
subprocess.run([*common, '--preflight', '--model', 'gemma4-e4b', '--run-name', E4B_RUN], check=True)

In [ ]:
subprocess.run([
    *common, '--model', 'gemma4-e4b', '--run-name', E4B_RUN,
    '--epochs', '2', '--lora-rank', '16', '--seed', '560',
    '--resume', 'auto', '--export-merged-hf', '--export-gguf', '--quantization', 'Q8_0',
], check=True)

## Smaller E2B fallback
Use this candidate only after E4B fails the accelerator memory preflight. The rank-8 attention-only adapter is the smallest documented memory fallback that preserves the complete dataset, both epochs, no-truncation gate, response-only loss, seed, and evaluation contract. It still requires BF16-capable CUDA hardware; the CLI rejects a T4 before model download. It writes to a separate run directory and its adapter scope is bound into every run manifest.

In [ ]:
E2B_RUN = 'gemma4-e2b-r8-e2-seed560-attention'
subprocess.run([
    *common, '--preflight', '--model', 'gemma4-e2b', '--run-name', E2B_RUN,
    '--epochs', '2', '--lora-rank', '8', '--lora-targets', 'attention', '--seed', '560',
], check=True)

In [ ]:
subprocess.run([
    *common, '--model', 'gemma4-e2b', '--run-name', E2B_RUN,
    '--epochs', '2', '--lora-rank', '8', '--lora-targets', 'attention', '--seed', '560',
    '--resume', 'auto', '--export-merged-hf', '--export-gguf', '--quantization', 'Q8_0',
], check=True)

Use the fallback only after validation shows overfitting or a material regression. Its separate run name prevents it from overwriting the primary candidate.

In [ ]:
RUN_E4B_FALLBACK = False
if RUN_E4B_FALLBACK:
    fallback_common = [
        *common, '--model', 'gemma4-e4b', '--run-name', 'gemma4-e4b-r8-e1-seed560',
        '--epochs', '1', '--lora-rank', '8', '--seed', '560',
    ]
    subprocess.run([*fallback_common, '--preflight'], check=True)
    subprocess.run([
        *fallback_common,
        '--resume', 'auto', '--export-merged-hf', '--export-gguf', '--quantization', 'Q8_0',
    ], check=True)

## Gemma 3 4B compatibility candidate
Use this smaller candidate when Gemma 4 E2B cannot pass the complete-sequence memory preflight on a BF16-capable accelerator. It keeps the identical corpus, prompt, RAG payload, response-only loss, seed, and promotion gates. Live T4 probes showed that Unsloth uses float32 attention there and cannot fit the longest example, so the CLI rejects non-BF16 hardware instead of truncating data.

In [ ]:
GEMMA3_RUN = 'gemma3-4b-r16-e2-seed560'
subprocess.run([
    *common, '--preflight', '--model', 'gemma3-4b', '--run-name', GEMMA3_RUN,
    '--epochs', '2', '--lora-rank', '16', '--lora-targets', 'all', '--seed', '560',
], check=True)

In [ ]:
subprocess.run([
    *common, '--model', 'gemma3-4b', '--run-name', GEMMA3_RUN,
    '--epochs', '2', '--lora-rank', '16', '--lora-targets', 'all', '--seed', '560',
    '--resume', 'auto', '--export-merged-hf', '--export-gguf', '--quantization', 'Q8_0',
], check=True)

## Optional 12B candidate
Leave this disabled until the E4B validation is complete. Enabling it performs preflight only. Start a 12B training run afterward only when preflight proves that the full sequence fits the assigned free accelerator.

In [ ]:
PREFLIGHT_12B = False
if PREFLIGHT_12B:
    subprocess.run([
        *common, '--preflight', '--model', 'gemma4-12b',
        '--run-name', 'gemma4-12b-r16-e2-seed560',
    ], check=True)

Completed Drive run folders contain `preflight_manifest.json`, resumable checkpoints, adapter weights, a temporary merged Hugging Face model, Q8_0 GGUF, training metrics, `checksums.json`, and `run_manifest.json`. Download only the selected candidate. Convert the merged model on Apple Silicon with the current MLX-LM converter, then serve it on `127.0.0.1`; do not expose the MLX server directly to the tunnel or internet.